# Label rules (written before searching, started <date>)

**Positive:** a news source names ___, ___ and ___. Requires a source_url.
**Severity:** high = ___ ; moderate = ___ .
**Negative:** searched ___ and ___, found nothing. Label `none`,
source text "no report found (unconfirmed)".
**Label date:** chosen from ___ , not from ___ .
**New place:** coordinates from ___ , checked against ___ .
**Spelling:** reuse existing names (Gomti Nagar, Indira Nagar, Ashiyana).

**Targets:** 8+ positive events, 8+ negative events, 3+ heavy-rain negatives.


In [9]:
import pandas as pd
from pathlib import Path

ROOT=Path.cwd()
while not (ROOT/".git").exists() and ROOT!=ROOT.parent:
    ROOT=ROOT.parent

path = ROOT / "data" / "labels" / "waterlogging_reports.csv"


In [6]:
#check cell that validates csv file
lab=pd.read_csv(ROOT/"data"/"labels"/"waterlogging_reports.csv")
problems=[]

need=["date", "location_name", "lat", "lon", "severity", "source", "source_url"]
missing=[c for c in need if c not in lab.columns]
if missing:
    problems.append(f"missing columns: {missing}")
else:
    if pd.to_datetime(lab["date"], errors="coerce").isna().any():
        problems.append("unparseable dates")
    bad = lab[~lab["severity"].isin(["high", "moderate", "none"])]
    if len(bad): problems.append(f"{len(bad)} rows with unknown severity")
    out = lab[~lab["lat"].between(26.70, 27.00) | ~lab["lon"].between(80.80, 81.10)]
    if len(out): problems.append(f"{len(out)} rows outside the Lucknow box")
    dup = lab.duplicated(["date", "location_name"]).sum()
    if dup: problems.append(f"{dup} duplicate date+place rows")
    pos = lab[lab["severity"] != "none"]
    nourl = pos[pos["source_url"].isna() | (pos["source_url"].astype(str).str.strip() == "")]
    if len(nourl): problems.append(f"{len(nourl)} positives without a source_url")

print("OK" if not problems else "\n".join(problems))
print(lab.groupby("date")["severity"].agg(lambda s: s.value_counts().to_dict()))



OK
date
2022-07-21        {'none': 9}
2024-07-07    {'moderate': 5}
2024-09-17        {'none': 9}
2025-08-13        {'none': 9}
2026-09-25        {'high': 7}
Name: severity, dtype: object


In [10]:
lab= pd.read_csv(path)
new=lab[lab["date"]=="2022-07-21"].copy() #an existing negative
new["date"]= "2022-07-30"
new["severity"]="none"
new["source"] = "no report found (unconfirmed)"
new["source_url"] = ""

lab=pd.concat([lab,new], ignore_index=True)
lab.to_csv(path, index=False)

In [11]:
chk = pd.read_csv(path)
print(len(chk[chk["date"] == "2022-07-30"]))   # expect 9
print(chk[chk["date"] == "2022-07-30"][["date", "location_name", "severity", "source"]])

9
          date location_name severity                         source
39  2022-07-30    Hazratganj     none  no report found (unconfirmed)
40  2022-07-30   Gomti Nagar     none  no report found (unconfirmed)
41  2022-07-30       Aliganj     none  no report found (unconfirmed)
42  2022-07-30      Charbagh     none  no report found (unconfirmed)
43  2022-07-30         Chowk     none  no report found (unconfirmed)
44  2022-07-30  Indira Nagar     none  no report found (unconfirmed)
45  2022-07-30      Alambagh     none  no report found (unconfirmed)
46  2022-07-30      Ashiyana     none  no report found (unconfirmed)
47  2022-07-30  Faizullaganj     none  no report found (unconfirmed)
